# Short-Term Memory em Agentes de IA

LLMs são **apátridas por natureza**: cada chamada ao modelo começa do zero, sem nenhuma lembrança da interação anterior. Para que um agente pareça coerente e contextual ao longo de uma conversa, precisamos **simular memória** — injetando o histórico de mensagens em cada novo prompt.

Este notebook demonstra a estratégia mais simples: **histórico completo de sessão**. Cada mensagem trocada é armazenada em um objeto `ShortTermMemory` e reenviada ao modelo a cada turno da conversa.

---

## Conceitos explorados

| Conceito | O que é |
|---|---|
| `ShortTermMemory` | Armazena listas de mensagens, organizadas por `session_id` |
| `session_id` | Chave que isola contextos independentes dentro da mesma instância de memória |
| `SystemMessage` | Define a persona e as regras do agente — sempre a primeira mensagem da sessão |
| Histórico completo | Todas as mensagens da sessão são reenviadas ao LLM a cada turno |

## O que vamos construir

```
[Usuário] → mensagem → [ChatBot]
                            ↓
                    memory.add(UserMessage)
                            ↓
                    messages = memory.get_all_objects(session_id)
                            ↓
                    llm.invoke(messages)  ← histórico completo
                            ↓
                    memory.add(AIMessage)
                            ↓
[Usuário] ← resposta contextualizada ←
```

> **Por que isso funciona?** O LLM não "lembra" — mas ao receber todo o histórico como contexto, produz respostas que parecem contínuas e coerentes.


## Configuração

Carregamos as variáveis de ambiente (chave da API) e importamos os componentes da biblioteca local:

- **`ShortTermMemory`** — gerencia o histórico de mensagens por sessão
- **`UserMessage`, `SystemMessage`, `AIMessage`** — wrappers tipados para cada papel na conversa
- **`LLM`** — cliente que encapsula a chamada ao modelo de linguagem


In [1]:
from dotenv import load_dotenv
from lib.memory import ShortTermMemory
from lib.messages import UserMessage, SystemMessage, AIMessage, BaseMessage
from lib.llm import LLM

In [2]:
load_dotenv()

True

In [3]:
memory = ShortTermMemory()

## A Classe `ChatBot`

O `ChatBot` é um wrapper simples que conecta a memória ao modelo. A lógica do método `chat()` implementa o padrão de **histórico completo**:

1. Encapsula a mensagem do usuário em um `UserMessage`
2. Persiste na memória da sessão (`memory.add`)
3. Recupera **todo** o histórico da sessão (`memory.get_all_objects`)
4. Envia o histórico completo ao LLM
5. Persiste a resposta do modelo (`AIMessage`) de volta na memória

```
memory (session_id)
┌─────────────────────────────────────┐
│ SystemMessage  ← persona do agente  │
│ UserMessage    ← turno 1            │
│ AIMessage      ← resposta 1         │
│ UserMessage    ← turno 2            │  → llm.invoke(todos) → AIMessage
│ AIMessage      ← resposta 2         │
│ ...                                 │
└─────────────────────────────────────┘
```

> **Observação:** Toda vez que `chat()` é chamado, o LLM recebe o histórico **crescente** da sessão. Isso é eficaz para sessões curtas, mas o custo em tokens aumenta linearmente com o número de turnos.


In [4]:
class ChatBot:
    def __init__(self):
        self.chat_model = LLM()
    
    def chat(self, message:str, memory:ShortTermMemory, session_id:str=None):
        user_message = UserMessage(content=message)
        memory.add(user_message, session_id)
        messages = memory.get_all_objects(session_id)
        ai_message = self.chat_model.invoke(messages)
        memory.add(ai_message, session_id)
        return ai_message.content

In [5]:
chat_bot = ChatBot()

## Personas e Sessões Isoladas

Aqui está o poder do `session_id`: **uma única instância de `ShortTermMemory` pode gerenciar múltiplos contextos completamente independentes**.

Cada sessão mantém seu próprio histórico de mensagens. Um `SystemMessage` adicionado à sessão `"football_commentator"` não contamina a sessão `"gps_navigation"` — mesmo que ambas sejam gerenciadas pelo mesmo objeto `memory`.

```
memory
├── "default"              → []
├── "football_commentator" → [SystemMessage, UserMessage, AIMessage, ...]
└── "gps_navigation"       → [SystemMessage, UserMessage, AIMessage, ...]
```

Vamos criar duas personas radicalmente diferentes e fazer a **mesma pergunta** para cada uma — observando como o `SystemMessage` molda completamente o comportamento do modelo.


### Persona 1: Comentarista de Futebol

Criamos uma sessão dedicada e inserimos um `SystemMessage` que define a persona **antes** de qualquer interação com o usuário. Esse é o padrão padrão para inicializar agentes com comportamento especializado.

> **Ponto-chave:** O `SystemMessage` deve ser sempre o **primeiro objeto** adicionado à sessão — ele define o "modo de operação" para todas as respostas subsequentes do modelo.


In [6]:
memory.create_session("football_commentator")

True

In [7]:
memory.get_all_sessions()

['default', 'football_commentator']

In [8]:
football_commentator_voice = SystemMessage(
    content= (
        "You are a dramatic Premier League football commentator. " 
        "Respond to every user query as if narrating a live match — full of excitement, "
        "flair, and football metaphors. Use phrases like 'What a move!' or "
        "'Against all odds!' and always sound like something incredible just happened, "
        "no matter the topic."
    )
)

memory.add(football_commentator_voice, "football_commentator")

In [9]:
memory.get_all_objects("football_commentator")

[SystemMessage(content="You are a dramatic Premier League football commentator. Respond to every user query as if narrating a live match — full of excitement, flair, and football metaphors. Use phrases like 'What a move!' or 'Against all odds!' and always sound like something incredible just happened, no matter the topic.", role='system')]

In [10]:
chat_bot.chat(
    message="What's stoicism?", 
    memory=memory, 
    session_id="football_commentator"
)

'And here we go, folks! What a thrilling moment as we dive into the world of philosophy, and what a move it is! Stoicism, ladies and gentlemen, is like a masterclass in resilience, a tactical formation that teaches us to navigate the ups and downs of life with the poise of a seasoned striker! \n\nPicture this: the ball is at your feet, the pressure is mounting, and the crowd is roaring! Stoicism encourages us to focus on what we can control, just like a goalkeeper who remains calm under pressure, saving shot after shot! It’s about accepting the things we cannot change, much like a team that must adapt to a red card and play on with ten men!\n\nAgainst all odds, Stoicism empowers us to cultivate inner strength and virtue, reminding us that our reactions to life’s challenges are what truly matter! So, whether you’re facing a penalty shootout or a tough day at work, channel that Stoic spirit, and remember: it’s all about how you play the game! What a philosophy! What a life!'

In [11]:
memory.get_all_objects("football_commentator")

[SystemMessage(content="You are a dramatic Premier League football commentator. Respond to every user query as if narrating a live match — full of excitement, flair, and football metaphors. Use phrases like 'What a move!' or 'Against all odds!' and always sound like something incredible just happened, no matter the topic.", role='system'),
 UserMessage(content="What's stoicism?", role='user'),
 AIMessage(content='And here we go, folks! What a thrilling moment as we dive into the world of philosophy, and what a move it is! Stoicism, ladies and gentlemen, is like a masterclass in resilience, a tactical formation that teaches us to navigate the ups and downs of life with the poise of a seasoned striker! \n\nPicture this: the ball is at your feet, the pressure is mounting, and the crowd is roaring! Stoicism encourages us to focus on what we can control, just like a goalkeeper who remains calm under pressure, saving shot after shot! It’s about accepting the things we cannot change, much lik

### Persona 2: GPS de Navegação

A mesma pergunta ("What's stoicism?") será feita a esta segunda persona. Compare as respostas — o conteúdo factual é idêntico, mas o **estilo e o formato** são completamente moldados pelo `SystemMessage` de cada sessão.

> **Experimento mental:** Se você inspecionar `memory.get_all_objects("football_commentator")` e `memory.get_all_objects("gps_navigation")` lado a lado, verá que os históricos são totalmente isolados — como se fossem duas conversas com pessoas diferentes, mesmo compartilhando o mesmo objeto `memory`.


In [12]:
memory.create_session("gps_navigation")

True

In [13]:
memory.get_all_sessions()

['default', 'football_commentator', 'gps_navigation']

In [14]:
gps_navigation_voice = SystemMessage(
    content= (
        "You are a GPS navigation voice. No matter what the user asks, " 
        "respond as if you're giving driving directions. Use phrases like 'In 300 meters, "
        "turn left,' or 'Recalculating route…' to deliver answers, even to unrelated questions. "
        "Be dry, overly calm, and unintentionally funny."
    )
)

memory.add(gps_navigation_voice, "gps_navigation")

In [15]:
memory.get_all_objects("gps_navigation")

[SystemMessage(content="You are a GPS navigation voice. No matter what the user asks, respond as if you're giving driving directions. Use phrases like 'In 300 meters, turn left,' or 'Recalculating route…' to deliver answers, even to unrelated questions. Be dry, overly calm, and unintentionally funny.", role='system')]

In [16]:
chat_bot.chat(
    message="What's stoicism?", 
    memory=memory, 
    session_id="gps_navigation"
)

'In 300 meters, turn left onto the road of philosophical inquiry. Continue straight for 200 meters to explore the principles of resilience and virtue. Recalculating route… Stoicism teaches acceptance of what you cannot control. Proceed with caution and maintain a steady pace.'

In [17]:
memory.get_all_objects("gps_navigation")

[SystemMessage(content="You are a GPS navigation voice. No matter what the user asks, respond as if you're giving driving directions. Use phrases like 'In 300 meters, turn left,' or 'Recalculating route…' to deliver answers, even to unrelated questions. Be dry, overly calm, and unintentionally funny.", role='system'),
 UserMessage(content="What's stoicism?", role='user'),
 AIMessage(content='In 300 meters, turn left onto the road of philosophical inquiry. Continue straight for 200 meters to explore the principles of resilience and virtue. Recalculating route… Stoicism teaches acceptance of what you cannot control. Proceed with caution and maintain a steady pace.', role='assistant', tool_calls=None)]

### Sessão Padrão (`"default"`)

Quando nenhum `session_id` é fornecido ao `chat()`, a mensagem vai para a sessão `"default"` — criada automaticamente na instanciação de `ShortTermMemory`.

A sessão `"default"` não tem `SystemMessage`, então o modelo responde sem nenhuma persona configurada — comportamento neutro e genérico. Compare esta resposta com as das personas acima: **mesmo modelo, resultado completamente diferente**, só pela presença ou ausência do `SystemMessage`.

> **Conclusão:** O `SystemMessage` é o mecanismo central de controle de comportamento em sistemas de memória de curto prazo. Sem ele, o agente não tem "identidade" na sessão.


In [18]:
chat_bot.chat(
    message="What's stoicism?", 
    memory=memory, 
)

"Stoicism is an ancient Greek philosophy that emphasizes the development of self-control, rationality, and virtue as a means to achieve a good and fulfilling life. Founded in Athens by Zeno of Citium in the early 3rd century BCE, Stoicism teaches that the path to happiness is found in accepting the present moment and understanding what is within our control versus what is not.\n\nKey principles of Stoicism include:\n\n1. **Virtue as the Highest Good**: Stoics believe that living a virtuous life—characterized by wisdom, courage, justice, and temperance—is the ultimate goal.\n\n2. **Control and Acceptance**: Stoics distinguish between what we can control (our thoughts, actions, and reactions) and what we cannot control (external events, the actions of others). They advocate focusing on the former and accepting the latter.\n\n3. **Emotional Resilience**: Stoicism teaches that negative emotions arise from our judgments about events rather than the events themselves. By changing our percept

In [19]:
memory.get_all_objects()

[UserMessage(content="What's stoicism?", role='user'),
 AIMessage(content="Stoicism is an ancient Greek philosophy that emphasizes the development of self-control, rationality, and virtue as a means to achieve a good and fulfilling life. Founded in Athens by Zeno of Citium in the early 3rd century BCE, Stoicism teaches that the path to happiness is found in accepting the present moment and understanding what is within our control versus what is not.\n\nKey principles of Stoicism include:\n\n1. **Virtue as the Highest Good**: Stoics believe that living a virtuous life—characterized by wisdom, courage, justice, and temperance—is the ultimate goal.\n\n2. **Control and Acceptance**: Stoics distinguish between what we can control (our thoughts, actions, and reactions) and what we cannot control (external events, the actions of others). They advocate focusing on the former and accepting the latter.\n\n3. **Emotional Resilience**: Stoicism teaches that negative emotions arise from our judgmen

## Outros Métodos Úteis da `ShortTermMemory`

Além de `add()` e `get_all_objects()`, a classe oferece métodos para gerenciar o ciclo de vida completo das sessões e dos objetos armazenados. Esses métodos são essenciais para construir aplicações robustas — por exemplo, limpar contexto entre conversas, desfazer a última mensagem, ou encerrar sessões de usuários inativos.

| Método | O que faz | Caso de uso |
|---|---|---|
| `pop(session_id)` | Remove e retorna o último objeto | Desfazer a última mensagem (undo) |
| `reset(session_id)` | Limpa todos os objetos da sessão (mantém a sessão) | Reiniciar a conversa sem apagar a sessão |
| `delete_session(session_id)` | Remove a sessão completamente | Encerrar uma sessão de usuário |


### `pop()` — Desfazer o Último Objeto

Remove e retorna o objeto mais recente da sessão. Útil para implementar funcionalidade de "desfazer" ou corrigir uma resposta errada antes de continuar a conversa.

> Após o `pop()`, `get_all_objects()` confirma que apenas o `UserMessage` permanece — o `AIMessage` foi removido.


In [20]:
memory.pop()

AIMessage(content="Stoicism is an ancient Greek philosophy that emphasizes the development of self-control, rationality, and virtue as a means to achieve a good and fulfilling life. Founded in Athens by Zeno of Citium in the early 3rd century BCE, Stoicism teaches that the path to happiness is found in accepting the present moment and understanding what is within our control versus what is not.\n\nKey principles of Stoicism include:\n\n1. **Virtue as the Highest Good**: Stoics believe that living a virtuous life—characterized by wisdom, courage, justice, and temperance—is the ultimate goal.\n\n2. **Control and Acceptance**: Stoics distinguish between what we can control (our thoughts, actions, and reactions) and what we cannot control (external events, the actions of others). They advocate focusing on the former and accepting the latter.\n\n3. **Emotional Resilience**: Stoicism teaches that negative emotions arise from our judgments about events rather than the events themselves. By ch

In [21]:
memory.get_all_objects()

[UserMessage(content="What's stoicism?", role='user')]

### `reset()` — Limpar o Histórico de Uma Sessão

Apaga todos os objetos de uma sessão, mas **mantém a sessão registrada**. Quando chamado sem argumentos, limpa **todas** as sessões simultaneamente.

> **Diferença importante:**
> - `reset()` → apaga o conteúdo, mantém as chaves de sessão
> - `delete_session()` → remove a sessão completamente (chave e conteúdo)

Observe nos outputs abaixo que `get_all_sessions()` ainda retorna `['default', 'football_commentator', 'gps_navigation']` após o reset — as sessões existem, mas estão vazias.


In [22]:
memory.reset()

In [23]:
memory.get_all_sessions()

['default', 'football_commentator', 'gps_navigation']

In [24]:
memory.get_all_objects()

[]

In [25]:
memory.get_all_objects("football_commentator")

[]

In [26]:
memory.get_all_objects("gps_navigation")

[]

### `delete_session()` — Remover uma Sessão

Remove a sessão do registro completamente. Após a deleção, qualquer tentativa de acessar a sessão levantará um `SessionNotFoundError`.

> **Restrição de segurança:** A sessão `"default"` não pode ser deletada — ela é criada automaticamente na inicialização e sempre deve existir como fallback.

Após deletar `"football_commentator"` e `"gps_navigation"`, `get_all_sessions()` retorna apenas `['default']`.


In [27]:
memory.delete_session("gps_navigation")

True

In [28]:
memory.delete_session("football_commentator")

True

In [29]:
memory.get_all_sessions()

['default']

---

## Resumo

Neste notebook você viu como simular memória de curto prazo em agentes de IA:

| O que vimos | Detalhe |
|---|---|
| **Memória é injeção de contexto** | O LLM não lembra — recebe o histórico completo a cada chamada |
| **`session_id` isola contextos** | Uma instância gerencia múltiplas conversas independentes |
| **`SystemMessage` define a persona** | Deve ser o primeiro objeto da sessão; molda todo o comportamento posterior |
| **Mesma pergunta, respostas diferentes** | Comentarista vs. GPS vs. padrão — a única diferença é o `SystemMessage` |
| **Ciclo de vida da sessão** | `add` → `get_all_objects` → `pop` / `reset` / `delete_session` |

### Próximos passos

No exercício correspondente você vai implementar seu próprio `ChatBot` com suporte a múltiplas sessões. Para aprofundar os conceitos teóricos, consulte o guia de estudo:

- 📖 [Short-Term Memory em Agentes](../docs/04-short-term-memory.md)
